# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis and time window

**Unit of analysis:**
Each row represents one content page. I’m looking at pages individually so I can identify which ones may be worth prioritizing for a refresh.

**Time window:**
For this analysis, I’ll focus on the **March 2026 (`month = 2026-03`)** data slice. This gives me a consistent point in time to check the data and build the initial features.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



**Features:**
I’ll use signals that describe how each page is performing, including impressions, clicks, sessions, content age, CTR, and average position. These give the model useful information about the page’s current performance.

**Label:**
The proxy label will be `trend_direction`. I’ll treat pages with a value of **“down”** as pages showing signs of decline.

**Context:**
I’ll keep fields such as `content_id`, `client_id`, and `month` as context. They help identify the page and the time period, but they aren’t useful as predictive signals.

**Excluded:**
I’ll leave out `trend_direction` and any other fields that are directly derived from the outcome. Including them as features could give the model information about the answer it is supposed to predict.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face connection is ready.")

Hugging Face connection is ready.


In [5]:
Query 1
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_pages
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┐
│ total_rows │ unique_content_pages │
│   int64    │        int64         │
├────────────┼──────────────────────┤
│   78835655 │               427292 │
└────────────┴──────────────────────┘

In [9]:
con.sql(f"""
SELECT
    COUNT(*) AS rows_in_march,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬────────────┬────────────┐
│ rows_in_march │ first_date │ last_date  │
│     int64     │    date    │    date    │
├───────────────┼────────────┼────────────┤
│       9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────────┴────────────┴────────────┘

In [11]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │            3611061 │
└────────────┴────────────────────┘

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
all_columns = con.sql(f"""
SELECT column_name
FROM (
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
)
""").df()

print(all_columns.to_string(index=False))

             column_name
             report_date
          client_hash_id
         content_hash_id
          client_has_gsc
          client_has_ga4
      gsc_data_available
      ga4_data_available
         gsc_impressions
              gsc_clicks
        gsc_sum_position
        gsc_avg_position
           ga4_pageviews
            ga4_sessions
               ga4_users
    ga4_engaged_sessions
ga4_total_engagement_sec
        sessions_organic
         sessions_direct
       sessions_referral
         sessions_social
           sessions_paid
             sessions_ai
              ai_chatgpt
           ai_perplexity
               ai_gemini
              ai_copilot
               ai_claude
                 ai_meta
                ai_other
           scroll_events
                   month


In [16]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(scroll_events) AS scroll_events
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY client_hash_id, content_hash_id
LIMIT 10
""").df()

feature_frame

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,0.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.714744,0.0,0.0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.481453,4.0,0.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,9.0,1.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.0
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,48.0,0.0,14.753175,0.0,0.0
6,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,6.341880,1.0,0.0
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,67.0,0.0,12.791667,0.0,0.0
8,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,23.0,4.950311,8.0,0.0
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.0,50.390299,1.0,0.0


### When are these features available?

* **GSC impressions:** I can use this at the time of the decision because it shows how much search visibility the page has.
* **GSC clicks:** This is available because it shows how many clicks the page received from search.
* **GSC average position:** I can use this because it shows where the page is generally ranking in search results.
* **GA4 sessions:** This is available because it shows how many sessions the page received.
* **Scroll events:** I can use this as an engagement signal because it shows how users interacted with the page.


In [17]:
con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
LIMIT 1
""").df().T

,0
report_date,2026-03-01 00:00:00
client_hash_id,client_73cda7b4e4f265ea
content_hash_id,content_b7e512995f79d5a6
client_has_gsc,True
client_has_ga4,False
gsc_data_available,True
ga4_data_available,<NA>
gsc_impressions,20
gsc_clicks,0
gsc_sum_position,67


In [18]:
trend_frame = con.sql(f"""
WITH page_periods AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-07'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS first_7d_impressions,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-25'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS last_7d_impressions

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    *,
    CASE
        WHEN last_7d_impressions < first_7d_impressions
        THEN 'down'
        ELSE 'not_down'
    END AS trend_direction
FROM page_periods
LIMIT 10
""").df()

trend_frame

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,first_7d_impressions,last_7d_impressions,trend_direction
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,152.0,318.0,not_down
1,client_73cda7b4e4f265ea,content_05597932fe4da067,9.0,20.0,not_down
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,44.0,31.0,down
3,client_73cda7b4e4f265ea,content_05434271b257bb68,321.0,396.0,not_down
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,453.0,342.0,down
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,8.0,15.0,not_down
6,client_73cda7b4e4f265ea,content_2662845f598544ef,60.0,17.0,down
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,20.0,12.0,down
8,client_73cda7b4e4f265ea,content_712c365258cee05c,1180.0,1617.0,not_down
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,30.0,60.0,not_down


In [20]:
leak_test = trend_frame.copy()

# Deliberate leakage: this feature directly uses the label
leak_test["leaked_trend"] = leak_test["trend_direction"]

accuracy = (
    leak_test["leaked_trend"] == leak_test["trend_direction"]
).mean()

print(f"Score with leaked feature: {accuracy:.2%}")

Score with leaked feature: 100.00%


In [21]:
# Remove the deliberately leaked feature
honest_features = leak_test.drop(columns=["leaked_trend"])

print("Leaked feature removed.")
print("Features kept:", list(honest_features.columns))

Leaked feature removed.
Features kept: ['client_hash_id', 'content_hash_id', 'first_7d_impressions', 'last_7d_impressions', 'trend_direction']


In [22]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN is available:", HF_TOKEN is not None)

HF_TOKEN is available: True


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




One limitation I found is that the data is recorded daily, so I had to combine the daily records to look at each content page as a whole. Also, GSC and GA4 data are not available for every page. This means some of the information is incomplete and may not give us the full picture of how a page is performing.


## Self-check

* I explained what each row represents and which time period I’m using.
* I separated the features, label, context, and fields I’m leaving out.
* I used three queries to check that the data matches what I described.
* I checked whether GSC data was available using `IS TRUE`.
* I created five features that I can use at the time of the decision.
* I tested a feature that caused leakage and removed it afterward.
* I included a limitation that I found while working with the data.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.